<a href="https://colab.research.google.com/github/Alvinfaaiz/computer-vision-cnn-comparison/blob/main/Tugas_1_Observation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# Observasi Arsitektur CNN untuk Klasifikasi Objek
  
**Kelompok:** *15 October 2026*

**Muhammad Alvin Faa'iz (103012400229)**

**Nalendra Magi Jatayu (103012400362)**


**Dataset:** 5 kelas kendaraan (`bus`, `car`, `motorcycle`, `train`, `truck`) 
5.418 train / 709 validation / 708 test

| Arsitektur | Karakteristik utama |
|---|---|
| MobileNetV2 | Depthwise separable convolution, ringan dan cepat |
| ResNet50 | Residual/skip connection, mampu menangani jaringan sangat dalam |
| EfficientNetB0 | Compound scaling (depth, width, resolution) |
| VGG16 | Tumpukan conv 3×3 homogen, akurasi baik tetapi sangat berat |

Output: akurasi, precision, recall, F1-score, confusion matrix, waktu training,
waktu inferensi, jumlah parameter, grafik perbandingan, dan contoh prediksi model terbaik.


## Import & konfigurasi


In [ ]:
import os
import gc
import json
import time
import random
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import (
    MobileNetV2,
    ResNet50,
    EfficientNetB0,
    VGG16,
    mobilenet_v2,
    resnet50,
    efficientnet,
    vgg16,
)
from tensorflow.keras.utils import load_img

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

try:
    from IPython.display import display
except ImportError:  
    display = print

SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32  

HEAD_EPOCHS = 5
FINE_TUNE_EPOCHS = 5
HEAD_LR = 1e-3
FINE_TUNE_LR = 1e-5
DROPOUT = 0.3
DENSE_UNITS = 128

MODELS = [
    m.strip()
    for m in os.environ.get(
        "TUGAS_MODELS", "MobileNetV2,ResNet50,EfficientNetB0,VGG16"
    ).split(",")
    if m.strip()
]

QUICK_TEST = os.environ.get("TUGAS_QUICK_TEST", "0") == "1"
USE_MIXED_PRECISION = True

try:
    import google.colab 

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

ZIP_PATH = Path(
    "/content/drive/MyDrive/5_vehichles_for_classification/dataset.zip"
)
EXTRACT_PATH = Path("/content/dataset_kendaraan")

BASE_DIR = Path(
    os.environ.get("TUGAS_DATASET_DIR", str(EXTRACT_PATH / "dataset"))
)

if IN_COLAB:
    default_results = (
        "/content/drive/MyDrive/5_vehichles_for_classification/hasil_tugas1"
    )
else:
    default_results = "./hasil_tugas1"

RESULTS_DIR = Path(os.environ.get("TUGAS_RESULTS_DIR", default_results))
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

tf.keras.utils.set_random_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print("GPU terdeteksi:", [g.name for g in gpus])
else:
    print(
        "GPU TIDAK terdeteksi. Di Colab: Runtime > Change runtime type > T4 GPU."
    )

if USE_MIXED_PRECISION and gpus:
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
    print("Mixed precision: float16")
else:
    print("Mixed precision: tidak digunakan")

print("TensorFlow :", tf.__version__)
print("Dataset dir:", BASE_DIR)
print("Hasil dir  :", RESULTS_DIR)

## Persiapan dataset

In [ ]:
def ensure_dataset():
    if (BASE_DIR / "train").exists():
        print("Dataset sudah tersedia:", BASE_DIR)
        return

    if not ZIP_PATH.exists():
        raise FileNotFoundError(
            f"Dataset zip tidak ditemukan di {ZIP_PATH}. "
            "Sesuaikan ZIP_PATH, atau letakkan dataset langsung di BASE_DIR."
        )

    EXTRACT_PATH.mkdir(parents=True, exist_ok=True)
    print("Mengekstrak dataset...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)
    print("Ekstraksi selesai:", BASE_DIR)


ensure_dataset()

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp", ".tif", ".tiff"}

for split in ["train", "validation", "test"]:
    split_dir = BASE_DIR / split
    assert split_dir.exists(), f"Folder {split_dir} tidak ditemukan"
    n_files = sum(
        1
        for p in split_dir.glob("*/*")
        if p.is_file() and p.suffix.lower() in IMG_EXTS
    )
    print(f"{split:12s}: {n_files} gambar")


## Load dataset & eksplorasi data (EDA)


In [ ]:
def load_split(split, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        directory=str(BASE_DIR / split),
        labels="inferred",
        label_mode="categorical",
        color_mode="rgb",
        batch_size=BATCH_SIZE,
        image_size=IMG_SIZE,
        shuffle=shuffle,
        seed=SEED if shuffle else None,
    )


train_ds = load_split("train", True)
val_ds = load_split("validation", False)
test_ds = load_split("test", False)

CLASS_NAMES = list(train_ds.class_names)
assert list(val_ds.class_names) == CLASS_NAMES, "Urutan kelas validation berbeda"
assert list(test_ds.class_names) == CLASS_NAMES, "Urutan kelas test berbeda"
NUM_CLASSES = len(CLASS_NAMES)

print("Kelas        :", CLASS_NAMES)
print("Jumlah kelas :", NUM_CLASSES)
print(
    "Batch        :",
    f"train={len(train_ds)} | val={len(val_ds)} | test={len(test_ds)}",
)


### Distribusi jumlah gambar per kelas


In [ ]:
def count_files(split):
    counts = {}
    for cls in CLASS_NAMES:
        counts[cls] = sum(
            1
            for p in (BASE_DIR / split / cls).iterdir()
            if p.is_file() and p.suffix.lower() in IMG_EXTS
        )
    return counts


dist = pd.DataFrame(
    {
        "train": count_files("train"),
        "validation": count_files("validation"),
        "test": count_files("test"),
    }
)
dist.loc["TOTAL"] = dist.sum()
display(dist)

fig, ax = plt.subplots(figsize=(10, 4))
dist.drop(index="TOTAL").plot(kind="bar", rot=0, ax=ax)
ax.set_title("Distribusi jumlah gambar per kelas")
ax.set_ylabel("Jumlah gambar")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "distribusi_kelas.png", dpi=150, bbox_inches="tight")
plt.show()


### Contoh gambar dari data training


In [ ]:
def show_samples(ds, n=10):
    fig = plt.figure(figsize=(12, 6))
    shown = 0
    for images, labels in ds:
        for i in range(len(images)):
            if shown >= n:
                break
            ax = plt.subplot(2, 5, shown + 1)
            plt.imshow(images[i].numpy().astype("uint8"))
            plt.title(CLASS_NAMES[int(np.argmax(labels[i]))])
            plt.axis("off")
            shown += 1
        if shown >= n:
            break
    plt.suptitle("Contoh gambar training")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "contoh_gambar.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


show_samples(train_ds)


## Augmentasi & preprocessing


In [ ]:
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.08, fill_mode="nearest"),
        layers.RandomZoom(0.08, fill_mode="nearest"),
        layers.RandomTranslation(0.08, 0.08, fill_mode="nearest"),
        layers.RandomContrast(0.10),
    ],
    name="augmentasi",
)


def prepare_ds(ds, preprocess_fn, augment=False):
    if augment:
        ds = ds.map(
            lambda x, y: (data_augmentation(x, training=True), y),
            num_parallel_calls=tf.data.AUTOTUNE,
        )
        ds = ds.shuffle(1000, seed=SEED, reshuffle_each_iteration=True)

    ds = ds.map(
        lambda x, y: (preprocess_fn(x), y),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    return ds.prefetch(tf.data.AUTOTUNE)


for images, _ in train_ds.take(1):
    fig = plt.figure(figsize=(12, 3))
    for i in range(5):
        augmented = data_augmentation(images, training=True)
        ax = plt.subplot(1, 5, i + 1)
        plt.imshow(
            np.clip(augmented[0].numpy(), 0, 255).astype("uint8")
        )
        plt.axis("off")
    plt.suptitle("Contoh augmentasi")
    plt.tight_layout()
    plt.savefig(
        RESULTS_DIR / "contoh_augmentasi.png", dpi=150, bbox_inches="tight"
    )
    plt.show()
    plt.close(fig)
    break


## Arsitektur


In [ ]:
ARCHITECTURES = {
    "MobileNetV2": {
        "builder": MobileNetV2,
        "preprocess": mobilenet_v2.preprocess_input,
        "unfreeze": 30,
        "desc": "Depthwise separable convolution; ringan dan cepat.",
    },
    "ResNet50": {
        "builder": ResNet50,
        "preprocess": resnet50.preprocess_input,
        "unfreeze": 20,
        "desc": "Residual connection untuk jaringan sangat dalam.",
    },
    "EfficientNetB0": {
        "builder": EfficientNetB0,
        "preprocess": efficientnet.preprocess_input,
        "unfreeze": 30,
        "desc": "Compound scaling yang efisien.",
    },
    "VGG16": {
        "builder": VGG16,
        "preprocess": vgg16.preprocess_input,
        "unfreeze": 4,
        "desc": "Conv 3x3 homogen; komputasi berat meski parameter feature extractor lebih sedikit.",
    },
}


def build_model(name):
    config = ARCHITECTURES[name]

    base = config["builder"](
        include_top=False,
        weights="imagenet",
        input_shape=(*IMG_SIZE, 3),
    )
    base.trainable = False

    inputs = layers.Input(shape=(*IMG_SIZE, 3), name="input")

    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
    x = layers.Dropout(DROPOUT, name="dropout_1")(x)
    x = layers.Dense(DENSE_UNITS, activation="relu", name="fc_128")(x)
    x = layers.Dropout(DROPOUT / 2, name="dropout_2")(x)
    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        dtype="float32",
        name="prediksi",
    )(x)

    model = keras.Model(inputs, outputs, name=name)
    return model, base, config["preprocess"]


def compile_model(model, learning_rate):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )


def unfreeze_top(base, n_layers):
    """Unfreeze n layer terakhir; BatchNorm tetap dibekukan agar stabil."""
    base.trainable = True
    for layer in base.layers[:-n_layers]:
        layer.trainable = False
    for layer in base.layers[-n_layers:]:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

    n_trainable = sum(1 for layer in base.layers if layer.trainable)
    print(f"  Fine-tuning: {n_trainable}/{len(base.layers)} layer base trainable")


## Training & evaluasi semua arsitektur

In [ ]:
results_path = RESULTS_DIR / "results.csv"
y_true_path = RESULTS_DIR / "y_true.npy"
test_paths_path = RESULTS_DIR / "test_paths.json"


def dataset_file_paths(ds, split):
    """Ambil path file sesuai urutan dataset (shuffle=False)."""
    if hasattr(ds, "file_paths"):
        return list(ds.file_paths)

    print("Peringatan: file_paths tidak tersedia, urutan path direkonstruksi.")
    paths = []
    for cls in CLASS_NAMES:
        cls_dir = BASE_DIR / split / cls
        paths.extend(
            str(p)
            for p in sorted(cls_dir.iterdir())
            if p.is_file() and p.suffix.lower() in IMG_EXTS
        )
    return paths


TEST_PATHS = dataset_file_paths(test_ds, "test")
Y_TRUE = np.concatenate(
    [np.argmax(y.numpy(), axis=1) for _, y in test_ds]
).astype(int)

np.save(y_true_path, Y_TRUE)
with open(test_paths_path, "w") as f:
    json.dump(TEST_PATHS, f, indent=2)

assert len(TEST_PATHS) == len(Y_TRUE), (len(TEST_PATHS), len(Y_TRUE))
print("Jumlah data test:", len(Y_TRUE))

results = []
if results_path.exists():
    results = pd.read_csv(results_path).to_dict("records")


def artifacts_complete(name):
    return (
        (RESULTS_DIR / f"best_{name}.weights.h5").exists()
        and (RESULTS_DIR / f"probs_{name}.npy").exists()
        and any(row["model"] == name for row in results)
    )


def measure_inference_ms(model, ds, n_images):
    """Waktu inferensi model saja (ms/gambar), tanpa I/O dataset."""
    batch_x, _ = next(iter(ds))
    _ = model(batch_x, training=False).numpy()

    n_repeat = max(1, int(np.ceil(n_images / len(batch_x))))
    start = time.perf_counter()
    for _ in range(n_repeat):
        _ = model(batch_x, training=False).numpy()
    elapsed = time.perf_counter() - start
    return elapsed / (n_repeat * len(batch_x)) * 1000.0


def merge_history(h1, h2):
    keys = set(h1.history) | set(h2.history)
    merged = {
        key: [float(v) for v in h1.history.get(key, []) + h2.history.get(key, [])]
        for key in keys
    }
    merged["stage_split"] = len(h1.history["loss"])
    return merged


def plot_history(history, name):
    epochs = range(1, len(history["loss"]) + 1)
    split = history["stage_split"]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, metric, title in zip(
        axes, ["accuracy", "loss"], ["Akurasi", "Loss"]
    ):
        ax.plot(epochs, history[metric], "o-", label="train")
        ax.plot(epochs, history[f"val_{metric}"], "s-", label="validation")
        ax.axvline(
            split + 0.5,
            color="gray",
            linestyle="--",
            label="mulai fine-tuning",
        )
        ax.set_title(f"{title} — {name}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(title)
        ax.grid(alpha=0.3)
        ax.legend()

    plt.tight_layout()
    plt.savefig(
        RESULTS_DIR / f"kurva_{name}.png", dpi=150, bbox_inches="tight"
    )
    plt.show()
    plt.close(fig)


In [ ]:
for name in MODELS:
    print("\n" + "=" * 80)
    print(f"MODEL: {name} — {ARCHITECTURES[name]['desc']}")
    print("=" * 80)

    if artifacts_complete(name):
        print("Hasil sudah lengkap, dilewati. Hapus artefaknya untuk mengulang.")
        continue

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)

    model, base, preprocess = build_model(name)
    train_model_ds = prepare_ds(train_ds, preprocess, augment=True)
    val_model_ds = prepare_ds(val_ds, preprocess)
    test_model_ds = prepare_ds(test_ds, preprocess)

    if QUICK_TEST:
        train_model_ds = train_model_ds.take(20)
        val_model_ds = val_model_ds.take(5)

    print(f"Total parameter : {model.count_params():,}")
    print(f"Mulai training  : {time.strftime('%H:%M:%S')}")

    head_weights = RESULTS_DIR / f"best_{name}_head.weights.h5"
    ft_weights = RESULTS_DIR / f"best_{name}_ft.weights.h5"

    callbacks_head = [
        keras.callbacks.ModelCheckpoint(
            filepath=str(head_weights),
            monitor="val_accuracy",
            mode="max",
            save_best_only=True,
            save_weights_only=True,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=2,
            restore_best_weights=False,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=1,
            min_lr=1e-6,
            verbose=1,
        ),
    ]

    compile_model(model, HEAD_LR)
    start_time = time.perf_counter()

    h1 = model.fit(
        train_model_ds,
        validation_data=val_model_ds,
        epochs=1 if QUICK_TEST else HEAD_EPOCHS,
        callbacks=callbacks_head,
        verbose=1,
    )

    unfreeze_top(base, ARCHITECTURES[name]["unfreeze"])
    compile_model(model, FINE_TUNE_LR)

    callbacks_ft = [
        keras.callbacks.ModelCheckpoint(
            filepath=str(ft_weights),
            monitor="val_accuracy",
            mode="max",
            save_best_only=True,
            save_weights_only=True,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=2,
            restore_best_weights=False,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=1,
            min_lr=1e-7,
            verbose=1,
        ),
    ]

    h2 = model.fit(
        train_model_ds,
        validation_data=val_model_ds,
        epochs=1 if QUICK_TEST else FINE_TUNE_EPOCHS,
        callbacks=callbacks_ft,
        verbose=1,
    )
    training_seconds = time.perf_counter() - start_time

    # ---------------- Pilih bobot terbaik dari kedua tahap ----------------
    best_head_acc = max(h1.history["val_accuracy"])
    best_ft_acc = max(h2.history["val_accuracy"])
    if best_ft_acc >= best_head_acc:
        best_stage_weights, best_val_acc = ft_weights, best_ft_acc
    else:
        best_stage_weights, best_val_acc = head_weights, best_head_acc

    model.load_weights(best_stage_weights)

    final_weights = RESULTS_DIR / f"best_{name}.weights.h5"
    shutil.copyfile(best_stage_weights, final_weights)
    for path in [head_weights, ft_weights]:
        if path.exists():
            path.unlink()

    # ---------------- Evaluasi pada data test ----------------
    probs = model.predict(test_model_ds, verbose=0).astype("float32")
    y_pred = probs.argmax(axis=1)

    test_acc = accuracy_score(Y_TRUE, y_pred)
    p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
        Y_TRUE, y_pred, average="macro", zero_division=0
    )
    p_weighted, r_weighted, f_weighted, _ = precision_recall_fscore_support(
        Y_TRUE, y_pred, average="weighted", zero_division=0
    )

    report = classification_report(
        Y_TRUE,
        y_pred,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )
    print(report)

    np.save(RESULTS_DIR / f"probs_{name}.npy", probs)
    with open(RESULTS_DIR / f"report_{name}.txt", "w") as f:
        f.write(report)

    cm = confusion_matrix(Y_TRUE, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format="d"
    )
    ax.set_title(f"Confusion Matrix — {name}")
    plt.tight_layout()
    plt.savefig(
        RESULTS_DIR / f"cm_{name}.png", dpi=150, bbox_inches="tight"
    )
    plt.show()
    plt.close(fig)

    history = merge_history(h1, h2)
    with open(RESULTS_DIR / f"history_{name}.json", "w") as f:
        json.dump(history, f, indent=2)
    plot_history(history, name)

    inference_ms = measure_inference_ms(
        model, test_model_ds, len(TEST_PATHS)
    )
    weights_mb = final_weights.stat().st_size / (1024 ** 2)

    row = {
        "model": name,
        "params_m": model.count_params() / 1e6,
        "weights_mb": weights_mb,
        "train_time_s": training_seconds,
        "inference_ms_per_img": inference_ms,
        "best_val_acc": float(best_val_acc),
        "test_acc": float(test_acc),
        "precision_macro": float(p_macro),
        "recall_macro": float(r_macro),
        "f1_macro": float(f_macro),
        "precision_weighted": float(p_weighted),
        "recall_weighted": float(r_weighted),
        "f1_weighted": float(f_weighted),
    }

    results = [row_ for row_ in results if row_["model"] != name]
    results.append(row)
    pd.DataFrame(results).to_csv(results_path, index=False)

    print(
        f"\nSelesai {name}: test acc={test_acc:.4f}, F1={f_macro:.4f}, "
        f"training={training_seconds / 60:.1f} menit, "
        f"inference={inference_ms:.2f} ms/gambar"
    )

    del model, base
    gc.collect()

print("\nSemua model selesai. Hasil tersimpan di:", RESULTS_DIR)


## Tabel perbandingan

In [ ]:
results_df = pd.DataFrame(results)
if results_df.empty:
    raise RuntimeError("Belum ada hasil. Jalankan cell training terlebih dahulu.")

results_df = (
    results_df[results_df["model"].isin(MODELS)]
    .sort_values("test_acc", ascending=False)
    .reset_index(drop=True)
)

display_cols = [
    "model",
    "params_m",
    "weights_mb",
    "train_time_s",
    "inference_ms_per_img",
    "best_val_acc",
    "test_acc",
    "precision_macro",
    "recall_macro",
    "f1_macro",
]
table = results_df[display_cols].copy()

for col in [
    "best_val_acc",
    "test_acc",
    "precision_macro",
    "recall_macro",
    "f1_macro",
]:
    table[col] = (table[col] * 100).round(2).astype(str) + "%"

table["params_m"] = table["params_m"].round(2)
table["weights_mb"] = table["weights_mb"].round(1)
table["train_time_s"] = table["train_time_s"].round(1)
table["inference_ms_per_img"] = table["inference_ms_per_img"].round(2)

print("Tabel perbandingan (diurutkan berdasarkan akurasi test)")
display(table)
table.to_csv(RESULTS_DIR / "tabel_perbandingan.csv", index=False)


## Visualisasi perbandingan

In [ ]:
metric_cols = ["test_acc", "precision_macro", "recall_macro", "f1_macro"]
metric_labels = [
    "Accuracy",
    "Precision (macro)",
    "Recall (macro)",
    "F1-score (macro)",
]

fig, ax = plt.subplots(figsize=(12, 5))
results_df.set_index("model")[metric_cols].plot(kind="bar", rot=0, ax=ax)
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_ylabel("Skor")
ax.set_title("Perbandingan metrik evaluasi antar arsitektur")
ax.grid(axis="y", alpha=0.3)
ax.legend(metric_labels, loc="lower right")
plt.tight_layout()
plt.savefig(
    RESULTS_DIR / "perbandingan_metrik.png", dpi=150, bbox_inches="tight"
)
plt.show()
plt.close(fig)


In [ ]:
x_cols = ["params_m", "train_time_s", "inference_ms_per_img"]
x_labels = [
    "Parameter (juta)",
    "Waktu training (detik)",
    "Waktu inferensi (ms/gambar)",
]
titles = [
    "Kompleksitas (parameter) vs akurasi",
    "Waktu training vs akurasi",
    "Waktu inferensi vs akurasi",
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, x_col, x_label, title in zip(axes, x_cols, x_labels, titles):
    ax.scatter(results_df[x_col], results_df["test_acc"], s=80)
    for _, row in results_df.iterrows():
        ax.annotate(
            row["model"],
            (row[x_col], row["test_acc"]),
            textcoords="offset points",
            xytext=(6, 4),
            fontsize=8,
        )
    ax.set_xlabel(x_label)
    ax.set_ylabel("Akurasi test")
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))

plt.tight_layout()
plt.savefig(
    RESULTS_DIR / "perbandingan_efisiensi.png", dpi=150, bbox_inches="tight"
)
plt.show()
plt.close(fig)


In [ ]:
n_models = len(results_df)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 5))
if n_models == 1:
    axes = [axes]

for ax, name in zip(axes, results_df["model"]):
    probs = np.load(RESULTS_DIR / f"probs_{name}.npy")
    cm = confusion_matrix(Y_TRUE, probs.argmax(axis=1))
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format="d"
    )
    ax.set_title(name)

plt.suptitle("Confusion matrix semua arsitektur", y=1.02)
plt.tight_layout()
plt.savefig(
    RESULTS_DIR / "semua_confusion_matrix.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)


## Model terbaik & contoh prediksi

In [ ]:
best_row = results_df.iloc[0]
best_name = best_row["model"]
best_probs = np.load(RESULTS_DIR / f"probs_{best_name}.npy")
best_pred = best_probs.argmax(axis=1)

print(
    f"Model terbaik: {best_name} "
    f"(akurasi test {best_row['test_acc'] * 100:.2f}%)"
)
print(open(RESULTS_DIR / f"report_{best_name}.txt").read())

rng = np.random.default_rng(SEED)
correct_idx = np.where(best_pred == Y_TRUE)[0]
wrong_idx = np.where(best_pred != Y_TRUE)[0]

n_correct = min(8, len(correct_idx))
n_wrong = min(4, len(wrong_idx))
chosen = list(rng.choice(correct_idx, n_correct, replace=False))
chosen += list(rng.choice(wrong_idx, n_wrong, replace=False))

fig = plt.figure(figsize=(15, 9))
for i, idx in enumerate(chosen):
    ax = plt.subplot(3, 4, i + 1)
    ax.imshow(load_img(TEST_PATHS[idx], target_size=IMG_SIZE))
    color = "green" if best_pred[idx] == Y_TRUE[idx] else "red"
    ax.set_title(
        f"True: {CLASS_NAMES[Y_TRUE[idx]]}\n"
        f"Pred: {CLASS_NAMES[best_pred[idx]]}",
        color=color,
        fontsize=10,
    )
    ax.axis("off")

plt.suptitle(f"Contoh prediksi — {best_name}")
plt.tight_layout()
plt.savefig(
    RESULTS_DIR / "contoh_prediksi_model_terbaik.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)


## Ringkasan untuk laporan


In [ ]:
summary_path = RESULTS_DIR / "ringkasan_hasil.md"
try:
    table_markdown = table.to_markdown(index=False)
except Exception:
    table_markdown = table.to_string(index=False)

with open(summary_path, "w") as f:
    f.write("# Ringkasan Hasil Tugas 1\n\n")
    f.write(
        f"Dataset: {len(TEST_PATHS)} gambar test, {NUM_CLASSES} kelas "
        f"({', '.join(CLASS_NAMES)})\n\n"
    )
    f.write("## Tabel perbandingan\n\n")
    f.write(table_markdown)
    f.write("\n\n## File yang dihasilkan\n\n")
    for path in sorted(RESULTS_DIR.glob("*")):
        f.write(f"- `{path.name}`\n")

print("Ringkasan disimpan:", summary_path)
print(open(summary_path).read())


## Checklist kesesuaian dengan kriteria tugas

| Kriteria | Dipenuhi di |
|---|---|
| Minimal 3 arsitektur CNN | `MODELS` (4 arsitektur) |
| Preprocessing: resize, normalisasi, augmentasi | Bagian 1–3 dan fungsi `prepare_ds` |
| Akurasi, precision, recall, F1-score | Cell training + `report_<model>.txt` |
| Confusion matrix | `cm_<model>.png` dan `semua_confusion_matrix.png` |
| Perbandingan performa | Tabel + `perbandingan_metrik.png` |
| Efisiensi komputasi (waktu training/inferensi) | `perbandingan_efisiensi.png` |
| Visualisasi prediksi model terbaik | `contoh_prediksi_model_terbaik.png` |

**Catatan untuk laporan:** gunakan tabel perbandingan dan grafik di atas,
lalu jelaskan hubungan antara jumlah parameter/kedalaman arsitektur dengan
akurasi, waktu training, dan waktu inferensi. Sebutkan juga kelebihan dan
kekurangan masing-masing arsitektur.
